used vit in reference to classification

In [13]:
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torchvision
import torch.utils.data as dataloader

# Data Ingestion

In [14]:
# transformation for PIL->TENSOR

transformation_operation=torchvision.transforms.Compose([torchvision.transforms.ToTensor()])

In [15]:
# importing dataset
train_dataset= torchvision.datasets.MNIST(root="./data",train=True,download=True,transform=transformation_operation)

test_dataset= torchvision.datasets.MNIST(root="./data",train=False,download=True,transform=transformation_operation)

In [16]:
num_classes=10
batch_size=64
num_channels=1
image_size=28
patch_size=7
num_patches=(image_size//patch_size)**2
embedding_dim=64

attention_heads=4
mlp_hidden_nodes=128
transformer_blocks=4

learning_rate=0.001
epochs=5

In [17]:
num_patches

16

In [18]:
#define batches
train_loader=dataloader.DataLoader(dataset=train_dataset,batch_size=batch_size,shuffle=True)
val_loader=dataloader.DataLoader(dataset=test_dataset,batch_size=batch_size,shuffle=True)

### Part 1 : Patch Embedding

walkthrough

part1 : Patch embedding
part2 : Transformer encoder
part3 : transformer class

In [19]:
class PatchEmbedding(nn.Module):
  def __init__(self):
    super().__init__()
    self.patch_embed=nn.Conv2d(num_channels,embedding_dim,kernel_size=patch_size,stride=patch_size)
  def forward(self,x):
    #patch embedding
    x=self.patch_embed(x)
    #flatten
    x=x.flatten(2)
    # making it [64,16,64]
    x=x.transpose(1,2)
    return x

### Transformer encoder


In [20]:
class TransformerEncoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer1_norm1=nn.LayerNorm(embedding_dim)
    self.layer2_norm2=nn.LayerNorm(embedding_dim)
    self.multihead_attention=nn.MultiheadAttention(embedding_dim,attention_heads)
    self.mlp=nn.Sequential(
        nn.Linear(embedding_dim,mlp_hidden_nodes),
        nn.GELU(),
        nn.Linear(mlp_hidden_nodes,embedding_dim)
    )
  def forward(self,x):
    res1=x
    x=self.layer1_norm1(x)
    x, _ = self.multihead_attention(x, x, x)
    x = x + res1

    res2=x
    x=self.layer2_norm2(x)
    x=self.mlp(x)
    x=x+res2

    return x